# SkinAI — DS15 + ISIC 2019 혼합 학습 (Phase 3)

**목적**: AI Hub 합성 데이터(12,000장) + ISIC 2019 실제 임상 이미지(~11,569장) 혼합 학습으로  
합성 → 실제 이미지 도메인 갭 완화 (목표: 악성 클래스 Recall ≥ 60%, 전체 Top-1 ≥ 50%)

| 데이터 | 이미지 수 | 클래스 | 용도 |
|--------|----------|--------|------|
| AI Hub 08-15 합성 | 12,000 | 15종 전체 | 학습 |
| ISIC 2019 실제 임상 | ~11,569 | 8종 (DS15 매핑) | 학습 |
| ISIC 2019 val (15%) | ~2,042 | 8종 | **홀드아웃 평가만** |

> ⚠️ ISIC 2019에 없는 7종(보웬병·흑색점·사마귀·비립종·표피낭종·화농 육아종·피지샘증식증)은 AI Hub 합성 데이터만으로 학습됩니다.

In [1]:
# GPU / RAM 확인
!nvidia-smi
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print(f"시스템 RAM: {ram_gb:.1f} GB")

Tue May  5 09:13:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   29C    P0             48W /  600W |       3MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# ── 셀 1: 환경 감지 ────────────────────────────────────────────
import os
from pathlib import Path

try:
    import google.colab
    IS_COLAB = True
    COLAB_ROOT = "/content/colab_skin_ai"
    PROJECT_ROOT = COLAB_ROOT
except ImportError:
    IS_COLAB = False
    PROJECT_ROOT = str(Path.cwd())

print(f"환경        : {'Google Colab' if IS_COLAB else '로컬'}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

환경        : Google Colab
PROJECT_ROOT: /content/colab_skin_ai


In [3]:
# ── 셀 2: Drive 마운트 (Colab 전용) ────────────────────────────
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = "/content/drive/MyDrive/skin_ai"
else:
    print("로컬 환경 — Drive 마운트 건너뜀")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# ── 셀 3: 소스코드 clone / pull (Colab 전용) ────────────────────
if IS_COLAB:
    from dotenv import load_dotenv

    _env_path = f"{DRIVE_ROOT}/.env"
    if Path(_env_path).exists():
        load_dotenv(_env_path)
    else:
        print(f"경고: {_env_path} 없음 — GITHUB_TOKEN 없이 시도합니다")

    _token = os.getenv("GITHUB_TOKEN", "")
    _repo_url = (
        f"https://{_token}@github.com/kyoe-23/skin_ai.git"
        if _token else
        "https://github.com/kyoe-23/skin_ai.git"
    )

    if not Path(COLAB_ROOT).exists():
        !git clone {_repo_url} {COLAB_ROOT}
    else:
        # pull 전 결과 파일(untracked) 제거 — git에 커밋된 파일과 충돌 방지
        !git -C {COLAB_ROOT} clean -fd ai/results/
        !git -C {COLAB_ROOT} pull
else:
    print("로컬 환경 — 클론 건너뜀")

Removing ai/results/DS14_mixed/eval_aihub_val/
Removing ai/results/DS14_mixed/eval_dermnet_test/
Removing ai/results/DS14_mixed/loss_curve.png
Removing ai/results/DS14_mixed/training_log.json
Removing ai/results/DS15_mixed/loss_curve.png
Removing ai/results/DS15_mixed/training_log.json
Removing ai/results/thresholds.json
Updating 846c62c..c595a43
Fast-forward
 ai/results/DS14_mixed/DS14_mixed_report.md         | 126 ++++
 .../DS14_mixed/eval_aihub_val/confusion_matrix.png | Bin 0 -> 55048 bytes
 .../eval_aihub_val/evaluation_results.json         | 104 ++++
 .../DS14_mixed/eval_aihub_val/roc_curves.png       | Bin 0 -> 86168 bytes
 .../eval_dermnet_test/confusion_matrix.png         | Bin 0 -> 49128 bytes
 .../eval_dermnet_test/evaluation_results.json      | 104 ++++
 .../DS14_mixed/eval_dermnet_test/roc_curves.png    | Bin 0 -> 81383 bytes
 ai/results/DS14_mixed/loss_curve.png               | Bin 0 -> 94880 bytes
 ai/results/DS14_mixed/training_log.json            | 498 ++++++++++++++++

In [5]:
# ── 셀 4: 프로젝트 루트 이동 + 데이터 심링크 설정 ────────────────
os.chdir(PROJECT_ROOT)
print(f"현재 디렉토리: {os.getcwd()}")

if IS_COLAB:
    os.makedirs("data/processed", exist_ok=True)

    def _symlink(src: str, dst: str, label: str):
        # 공백 포함 경로 대응 — !ln 셸 명령 대신 os.symlink 사용
        src_path = Path(src)
        dst_path = Path(dst)
        if src_path.exists():
            if dst_path.is_symlink():
                dst_path.unlink()
            if not dst_path.exists():
                os.symlink(src_path, dst_path)
            print(f"  ✅ {dst} 심링크 완료")
        else:
            print(f"  ❌ {label} 없음 — {src}")

    # AI Hub DS15 원본 ZIP
    _symlink(f"{DRIVE_ROOT}/data/dataset_15", "data/dataset_15", "dataset_15")

    # ISIC 2019 원본 이미지 폴더 (폴더명에 공백 포함)
    _symlink(f"{DRIVE_ROOT}/data/ISIC 2019", "data/ISIC 2019", "ISIC 2019")

    # DS15 전처리 CSV
    _symlink(f"{DRIVE_ROOT}/data/processed/DS15", "data/processed/DS15", "DS15 전처리 CSV")

    # ISIC 2019 전처리 CSV (없으면 셀 7에서 생성)
    DRIVE_ISIC_CSV = f"{DRIVE_ROOT}/data/processed/isic2019"
    _symlink(DRIVE_ISIC_CSV, "data/processed/isic2019", "ISIC 전처리 CSV")
    if not Path(DRIVE_ISIC_CSV).exists():
        print("⚠️  ISIC 2019 전처리 CSV 없음 — 셀 7 실행 후 자동 생성됩니다")
    else:
        print("  (셀 7 건너뛸 수 있습니다)")

else:
    for d in ["data/dataset_15", "data/ISIC 2019", "data/processed/DS15", "data/processed/isic2019"]:
        status = "✅" if Path(d).exists() else "❌"
        print(f"{status} {d}")

!ls data/

현재 디렉토리: /content/colab_skin_ai
  ✅ data/dataset_15 심링크 완료
  ✅ data/ISIC 2019 심링크 완료
  ✅ data/processed/DS15 심링크 완료
  ✅ data/processed/isic2019 심링크 완료
  (셀 7 건너뛸 수 있습니다)
 dataset_14   dataset_15   dermnet  'ISIC 2019'   processed


In [6]:
# ── 셀 5: 패키지 설치 ───────────────────────────────────────────
!pip install -q \
    torch torchvision \
    pandas pillow tqdm \
    matplotlib python-dotenv scikit-learn

In [7]:
# ── 셀 6: 학습 전 체크리스트 ────────────────────────────────────
import torch
import pandas as pd
from pathlib import Path

checks = []
warnings = []

# 1. GPU
gpu_ok = torch.cuda.is_available()
checks.append(("GPU 사용 가능", gpu_ok))
if gpu_ok:
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU: {name} ({vram:.0f} GB)")

# 2. DS15 전처리 CSV
ds15_train = Path("data/processed/DS15/train.csv")
ds15_val   = Path("data/processed/DS15/val.csv")
ds15_ok = ds15_train.exists() and ds15_val.exists()
checks.append(("DS15 전처리 CSV (train/val)", ds15_ok))
if ds15_ok:
    n = len(pd.read_csv(ds15_train)) + len(pd.read_csv(ds15_val))
    print(f"  DS15 총 {n:,}건")

# 3. DS15 원본 ZIP 접근 가능 여부
ds15_raw = Path("data/dataset_15")
ds15_raw_ok = ds15_raw.exists()
checks.append(("DS15 원본 데이터 (data/dataset_15)", ds15_raw_ok))

# 4. ISIC 2019 원본 폴더
isic_raw = Path("data/ISIC 2019")
isic_raw_ok = isic_raw.exists()
checks.append(("ISIC 2019 원본 (data/ISIC 2019)", isic_raw_ok))
if isic_raw_ok:
    n_folders = sum(1 for p in isic_raw.iterdir() if p.is_dir())
    print(f"  ISIC 2019 클래스 폴더: {n_folders}개")

# 5. ISIC 2019 전처리 CSV (없으면 셀 7에서 생성)
isic_csv = Path("data/processed/isic2019/train.csv")
isic_csv_ok = isic_csv.exists()
checks.append(("ISIC 2019 전처리 CSV (data/processed/isic2019)", isic_csv_ok))
if not isic_csv_ok:
    warnings.append("  → 셀 7(ISIC 2019 전처리)을 실행하면 자동 생성됩니다.")

warnings.append("  ⚠️  ISIC 2019 미포함 7종은 AI Hub 합성 데이터만으로 학습됩니다.")
warnings.append("  ⚠️  악성 클래스(MEL/BCC/SCC) Recall ≥ 60% 달성 여부를 셀 10에서 확인하세요.")

print("\n" + "=" * 55)
print("학습 전 체크리스트")
print("=" * 55)
all_ok = True
for name, ok in checks:
    status = "✅" if ok else "❌"
    print(f"{status} {name}")
    if not ok:
        all_ok = False

print("\n주의사항:")
for w in warnings:
    print(w)

print("\n" + ("🟢 모든 필수 조건 충족" if all_ok else "🔴 위 항목을 먼저 해결하세요"))

  GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (102 GB)
  DS15 총 13,500건
  ISIC 2019 클래스 폴더: 8개

학습 전 체크리스트
✅ GPU 사용 가능
✅ DS15 전처리 CSV (train/val)
✅ DS15 원본 데이터 (data/dataset_15)
✅ ISIC 2019 원본 (data/ISIC 2019)
✅ ISIC 2019 전처리 CSV (data/processed/isic2019)

주의사항:
  ⚠️  ISIC 2019 미포함 7종은 AI Hub 합성 데이터만으로 학습됩니다.
  ⚠️  악성 클래스(MEL/BCC/SCC) Recall ≥ 60% 달성 여부를 셀 10에서 확인하세요.

🟢 모든 필수 조건 충족


In [ ]:
# ── 셀 7: ISIC 2019 전처리 (→ CSV) ──────────────────────────────
# Drive의 기존 CSV가 절대경로이면 자동 재생성 (Colab 환경 경로 불일치 방지)
import json as _json
import pandas as _pd
from pathlib import Path

ISIC_CSV_DIR = "data/processed/isic2019"
isic_train_csv = Path(f"{ISIC_CSV_DIR}/train.csv")

needs_regen = True
if isic_train_csv.exists():
    _sample = _pd.read_csv(isic_train_csv, nrows=1)
    if _sample["image_path"].iloc[0].startswith("/"):
        print("⚠️  절대경로 CSV 감지 — Colab 경로로 재생성합니다")
    else:
        needs_regen = False
        print(f"✅ 이미 존재 (상대경로) — {ISIC_CSV_DIR}/train.csv")

if needs_regen:
    print("ISIC 2019 전처리 시작 (약 1~2분 소요)...")
    !python -m ai.preprocessing.external_preprocessor \
        --root_dir "data/ISIC 2019" \
        --output_dir {ISIC_CSV_DIR} \
        --source isic2019 \
        --class_map_file ai/preprocessing/class_maps/isic2019_class_map.json \
        --class_idx_map_file ai/preprocessing/class_maps/unified_class_idx_map.json \
        --flat \
        --max_per_class 3000
    print("✅ 완료")

    # Drive에 저장 (Colab 전용)
    if IS_COLAB:
        import shutil
        _dst = f"{DRIVE_ROOT}/data/processed/isic2019"
        shutil.copytree(ISIC_CSV_DIR, _dst, dirs_exist_ok=True)
        print(f"✅ Drive 저장 완료: {_dst}")

# 결과 확인
if isic_train_csv.exists():
    _df = _pd.read_csv(isic_train_csv)
    print(f"\nISIC 2019 train: {len(_df):,}건")
    print(_df.groupby("class_name").size().to_string())

In [11]:
# ── 셀 8: DS15 + ISIC 2019 혼합 학습 실행 ──────────────────────
# AI Hub 합성 12,000 + ISIC 2019 실제 ~11,569 혼합
# WeightedRandomSampler: AI Hub weight=1.0, ISIC weight=1.5
# best 저장 기준: val_top1_acc (DS15 val.csv 기준)
#
# ── 재개 학습 (50→100 에폭) ──────────────────────────────────────
# epoch_50.pth에 scheduler_state_dict 미저장 → fast-forward 자동 적용
# 100에폭 기준 cosine 스케줄 50 스텝 위치 (LR ≈ 0.00024)에서 재개
RESUME_CKPT = "ai/results/DS15_mixed/epoch_50.pth"

from pathlib import Path
_resume_flag = f"--resume {RESUME_CKPT}" if Path(RESUME_CKPT).exists() else ""
_epochs = 100
print(f"Resume: {_resume_flag or '없음 (처음부터)'} | 목표 에폭: {_epochs}")

!EXTRA_DATA_DIR=data/processed/isic2019 \
 EXTERNAL_WEIGHT=1.5 \
 EXPERIMENT_NAME=ds15_mixed_isic \
 CHECKPOINT_DIR=ai/results/DS15_mixed \
 python -m ai.training.classifier.train \
     --backbone densenet121 \
     --data_dir data/processed/DS15 \
     --num_classes 15 \
     --num_epochs {_epochs} \
     --batch_size 64 \
     --root_dir {PROJECT_ROOT} \
     {_resume_flag}

Resume: --resume ai/results/DS15_mixed/epoch_50.pth | 목표 에폭: 100
INFO [INFO] CUDA 사용
피부질환 분류 모델 학습
  backbone    : densenet121
  device      : cuda
  epochs      : 100
  batch       : 64
  lr          : 0.0005
  warmup      : 3 epochs
  scheduler   : CosineAnnealingLR (T_max=97)
  num_classes : 15
  data_dir    : data/processed/DS15
INFO Dataset 로드: data/processed/DS15/train.csv (12000건, direction=None)
INFO Dataset 로드: data/processed/DS15/val.csv (1500건, direction=None)
INFO ExternalFacialDataset 로드: data/processed/isic2019/train.csv (11569건)
INFO ExternalFacialDataset 로드: data/processed/isic2019/val.csv (2042건)

  Train: 23569건 (AI Hub 12000 + 외부 11569)
  Val  : 3542건
INFO   Model     : DenseNet
INFO   Total     : 6,969,231 params
INFO   Trainable : 6,969,231 params
/usr/local/lib/python3.12/dist-packages/torch/optim/lr_scheduler.py:1195: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite ord

In [12]:
# ── 셀 9: 평가 1 — DS15 val (합성 성능 유지 확인) ─────────────────
# DS15 val.csv 기준 — 혼합 후에도 합성 val 성능이 유지되는지 확인
print("=" * 60)
print("평가 1: DS15 합성 val (성능 유지 확인)")
print("=" * 60)
!python -m ai.testing.evaluate \
    --checkpoint ai/results/DS15_mixed/best.pth \
    --data_dir data/processed/DS15 \
    --split val \
    --output_dir ai/results/DS15_mixed/eval_aihub_val \
    --root_dir {PROJECT_ROOT}

평가 1: DS15 합성 val (성능 유지 확인)
INFO [INFO] CUDA 사용
INFO Dataset 로드: data/processed/DS15/val.csv (1500건, direction=None)
평가 데이터: val.csv (1500건)

 SkinAI 분류 모델 평가 결과
 모델: densenet121
------------------------------------------------------------
 Top-1 Accuracy : 99.67%  가이드라인 목표(80%): 달성
 Top-3 Accuracy : 99.93%
 Macro F1-Score : 0.9967
 Macro AUC      : 0.9994
------------------------------------------------------------
 클래스              Prec   Recall       F1      AUC
------------------------------------------------------------
 광선각화증          1.0000   1.0000   1.0000   1.0000
 기저세포암          0.9901   1.0000   0.9950   1.0000
 멜라닌세포모반        0.9804   1.0000   0.9901   1.0000
 보웬병            1.0000   1.0000   1.0000   1.0000
 비립종            0.9901   1.0000   0.9950   1.0000
 사마귀            1.0000   0.9900   0.9950   1.0000
 악성흑색종          1.0000   0.9700   0.9848   0.9916
 지루각화증          1.0000   1.0000   1.0000   1.0000
 편평세포암          0.9901   1.0000   0.9950   1.0000
 표피낭종           1.

In [13]:
# ── 셀 10: 평가 2 — ISIC 2019 val (도메인 갭 측정) ───────────────
# ISIC 2019 val.csv (2,042장) — 학습에 사용하지 않은 홀드아웃 세트
# 목표: Top-1 ≥ 50%, 악성 클래스(MEL/BCC/SCC) Recall ≥ 60%
# 베이스라인(혼합 전): Top-1 26.40%, MEL 92.22% / BCC 0% / SCC 2.13%
print("=" * 60)
print("평가 2: ISIC 2019 val 홀드아웃 (도메인 갭 측정)")
print("베이스라인 → Top-1 26.40% | MEL 92% / BCC 0% / SCC 2%")
print("목표       → Top-1 ≥ 50% | 악성 3종 Recall ≥ 60%")
print("=" * 60)
!python -m ai.testing.evaluate \
    --checkpoint ai/results/DS15_mixed/best.pth \
    --data_dir data/processed/isic2019 \
    --split val \
    --output_dir ai/results/DS15_mixed/eval_isic_val

평가 2: ISIC 2019 val 홀드아웃 (도메인 갭 측정)
베이스라인 → Top-1 26.40% | MEL 92% / BCC 0% / SCC 2%
목표       → Top-1 ≥ 50% | 악성 3종 Recall ≥ 60%
INFO [INFO] CUDA 사용
INFO ExternalFacialDataset 로드: data/processed/isic2019/val.csv (2042건)
평가 데이터: val.csv (2042건)

 SkinAI 분류 모델 평가 결과
 모델: densenet121
------------------------------------------------------------
 Top-1 Accuracy : 77.72%  가이드라인 목표(80%): 미달
 Top-3 Accuracy : 95.10%
 Macro F1-Score : 0.4037
 Macro AUC      : 0.0000
------------------------------------------------------------
 클래스              Prec   Recall       F1      AUC
------------------------------------------------------------
 광선각화증          0.6772   0.6615   0.6693   0.9425
 기저세포암          0.8600   0.8733   0.8666   0.9799
 멜라닌세포모반        0.7576   0.8267   0.7906   0.9470
 보웬병            0.0000   0.0000   0.0000   0.0000
 비립종            0.0000   0.0000   0.0000   0.0000
 사마귀            0.0000   0.0000   0.0000   0.0000
 악성흑색종          0.7591   0.7422   0.7506   0.9327
 지루각화증          

In [14]:
# ── 셀 11: Threshold 최적화 ──────────────────────────────────
!python -m ai.testing.threshold_opt \
    --checkpoint ai/results/DS15_mixed/best.pth \
    --root_dir {PROJECT_ROOT}

INFO [INFO] CUDA 사용
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/colab_skin_ai/ai/testing/threshold_opt.py", line 248, in <module>
    main()
  File "/content/colab_skin_ai/ai/testing/threshold_opt.py", line 146, in main
    model.load_state_dict(checkpoint["model_state_dict"])
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 2635, in load_state_dict
    raise RuntimeError(
RuntimeError: Error(s) in loading state_dict for DenseNet:
	size mismatch for classifier.1.weight: copying a param with shape torch.Size([15, 1024]) from checkpoint, the shape in current model is torch.Size([6, 1024]).
	size mismatch for classifier.1.bias: copying a param with shape torch.Size([15]) from checkpoint, the shape in current model is torch.Size([6]).


In [15]:
# ── 셀 12: 체크포인트 Drive 저장 (Colab 전용) ────────────────────
# 런타임 종료 전 반드시 실행 — 저장하지 않으면 학습 결과 소실
if IS_COLAB:
    import shutil
    from pathlib import Path

    CKPT_SRC = f"{COLAB_ROOT}/ai/results/DS15_mixed"
    CKPT_DST = f"{DRIVE_ROOT}/ai/results/DS15_mixed"

    if not Path(CKPT_SRC).exists():
        print(f"❌ 체크포인트 없음: {CKPT_SRC}")
    else:
        shutil.copytree(CKPT_SRC, CKPT_DST, dirs_exist_ok=True)
        print(f"✅ Drive 저장 완료: {CKPT_DST}")
else:
    print("로컬 환경 — 체크포인트 이미 ai/results/DS15_mixed/ 에 저장됨")

✅ Drive 저장 완료: /content/drive/MyDrive/skin_ai/ai/results/DS15_mixed
